# Reading data

In [298]:
import pandas as pd

In [299]:
# California Housing Prices Dataset
df=pd.read_csv('housing.csv')
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


# Identify and separate the target variable

In [300]:
X=df.drop(columns=['median_house_value'])
y=df['median_house_value']

# Train/Test Split

In [301]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

# Clustering longitude and latitude (geographical) features to become a single categorical feature

In [302]:
from sklearn.cluster import KMeans
X_geo_col_names=['longitude','latitude']
kmeans_model=KMeans(n_clusters=10,random_state=42)
X_train['clusters']=kmeans_model.fit_predict(X_train[X_geo_col_names]).astype(str)
X_test['clusters']=kmeans_model.predict(X_test[X_geo_col_names]).astype(str)
X_train=X_train.drop(columns=['longitude','latitude'])
X_test=X_test.drop(columns=['longitude','latitude'])

# Separating numeric and categorical features

In [303]:
X_num_col_names=X_train.select_dtypes(include=['number']).columns
X_cat_col_names=X_train.select_dtypes(exclude=['number']).columns

# Visualizations of each feature vs the target

In [304]:
import plotly.express as px
from plotly.subplots import make_subplots

def plot_feature_target_relation(X,y):
    fig=make_subplots(rows=2,cols=4,subplot_titles=X.columns)
    for i,col_name in enumerate(X.columns):
        row,col=divmod(i,4)
        if col_name in X_num_col_names:
            trace=px.scatter(x=X[col_name],y=y).data[0]
        else:
            agg=pd.concat([X[col_name],y],axis=1).groupby(col_name).mean().reset_index()
            trace=px.bar(agg,x=col_name,y=y.name).data[0]
        fig.add_trace(trace,row=row+1,col=col+1)
    fig.update_layout(height=800)
    fig.show()
plot_feature_target_relation(X_train,y_train)
plot_feature_target_relation(X_test,y_test)

# Outliers handling

In [305]:
# numerical features outliers handling

q1=X_train[X_num_col_names].quantile(0.25)
q3=X_train[X_num_col_names].quantile(0.75)
iqr=q3-q1
lower_fence=q1-1.5*iqr
upper_fence=q3+1.5*iqr

X_train[X_num_col_names]=X_train[X_num_col_names].clip(lower=lower_fence,upper=upper_fence,axis=1)

# Missing values handling

In [306]:
from sklearn.impute import KNNImputer

# removal of rows with missing values at target
mask_y_train=~y_train.isnull()
mask_y_test=~y_test.isnull()
y_train=y_train[mask_y_train]
y_test=y_test[mask_y_test]
X_train=X_train[mask_y_train]
X_test=X_test[mask_y_test]

# removal of numeric features with high missing value ratio and removal of categorical features with moderate-high missing value ratio
num_mask_remove_features=(X_train[X_num_col_names].isnull().mean()>=0.5).reindex(X_train.columns,fill_value=False)
cat_mask_remove_features=(X_train[X_cat_col_names].isnull().mean()>=0.2).reindex(X_train.columns,fill_value=False)
mask_remove_features=num_mask_remove_features|cat_mask_remove_features
X_train=X_train.drop(columns=mask_remove_features[mask_remove_features].index)
X_test=X_test.drop(columns=mask_remove_features[mask_remove_features].index)
X_num_col_names=X_num_col_names.intersection(X_train.columns)
X_cat_col_names=X_cat_col_names.intersection(X_train.columns)

# imputation(numeric) of rows in low-moderate missing value ratio features
# (using knn imputer that uses train features and work on train/validation/test features)
knn_imputer=KNNImputer(n_neighbors=5)
X_train[X_num_col_names]=knn_imputer.fit_transform(X_train[X_num_col_names])
X_test[X_num_col_names]=knn_imputer.transform(X_test[X_num_col_names])

# imputation(categorical) of rows in low missing value ratio features
# (using mode imputation using mode of train features that is also used in imputation of train/validation/test features)
modes=X_train[X_cat_col_names].mode().iloc[0]
X_train[X_cat_col_names]=X_train[X_cat_col_names].fillna(modes)
X_test[X_cat_col_names]=X_test[X_cat_col_names].fillna(modes)

# Categorical encoding

In [307]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder=OneHotEncoder(sparse_output=False).set_output(transform="pandas")

X_train=pd.concat([X_train[X_num_col_names],one_hot_encoder.fit_transform(X_train[X_cat_col_names])],axis=1)
X_test=pd.concat([X_test[X_num_col_names],one_hot_encoder.transform(X_test[X_cat_col_names])],axis=1)

# Polynomial features

In [308]:
from sklearn.preprocessing import PolynomialFeatures

poly=PolynomialFeatures(degree=2,include_bias=False)

X_train_poly_only=pd.DataFrame(poly.fit_transform(X_train[X_num_col_names]),columns=poly.get_feature_names_out(X_num_col_names),index=X_train.index)
X_test_poly_only=pd.DataFrame(poly.transform(X_test[X_num_col_names]),columns=poly.get_feature_names_out(X_num_col_names),index=X_test.index)

X_num_col_names_poly=X_train_poly_only.columns

X_train_with_poly=pd.concat([X_train.drop(columns=X_num_col_names),X_train_poly_only],axis=1)
X_test_with_poly=pd.concat([X_test.drop(columns=X_num_col_names),X_test_poly_only],axis=1)

# Numerical Scaling

In [309]:
from sklearn.preprocessing import StandardScaler

standard_scaler=StandardScaler()

X_train[X_num_col_names]=standard_scaler.fit_transform(X_train[X_num_col_names])
X_test[X_num_col_names]=standard_scaler.transform(X_test[X_num_col_names])

X_train_with_poly[X_num_col_names_poly]=standard_scaler.fit_transform(X_train_with_poly[X_num_col_names_poly])
X_test_with_poly[X_num_col_names_poly]=standard_scaler.transform(X_test_with_poly[X_num_col_names_poly])

# Models Training

### Linear Regression

In [310]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,r2_score

linear_regression_model=LinearRegression()
linear_regression_model.fit(X_train,y_train)
y_pred=linear_regression_model.predict(X_test)

print(f'RMSE(Linear Regression): {root_mean_squared_error(y_test,y_pred)}')
print(f'R2 Score(Linear Regression): {r2_score(y_test,y_pred)}')

RMSE(Linear Regression): 70496.03106906386
R2 Score(Linear Regression): 0.621368492520368


### Polynomial Regression

In [311]:
# one of the team write this code
# (use X_train_with_poly,X_test_with_poly instead of X_train,X_test only here)

### Lasso Regression

In [312]:
# one of the team write this code
# (apply grid search for alpha parameter)

### Ridge Regression

In [313]:
# one of the team write this code
# (apply grid search for alpha parameter)